In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm

from model.dataset import BagsDataset, custom_collate_fn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
adata = sc.read_h5ad('../test.h5ad')

In [4]:
dataset = BagsDataset(adata, radius= 200, immune_cell='tcell')

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 6067.76it/s]

Total bags created: 3858
Average instances per bag: 8


In [5]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


In [70]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))

In [71]:
class Sparsemax(nn.Module):
    def __init__(self, dim=None):
        super(Sparsemax, self).__init__()
        self.dim = -1 if dim is None else dim

    def forward(self, input):
        input = input.transpose(0, self.dim)
        original_size = input.size()
        input = input.reshape(input.size(0), -1)
        input = input.transpose(0, 1)
        dim = 1

        number_of_logits = input.size(dim)

        input = input - torch.max(input, dim=dim, keepdim=True)[0].expand_as(input)

        zs = torch.sort(input=input, dim=dim, descending=True)[0]
        range = torch.arange(start=1, end=number_of_logits + 1, step=1, device=input.device, dtype=input.dtype).view(1, -1)
        range = range.expand_as(zs)

        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(input.type())
        k = torch.max(is_gt * range, dim, keepdim=True)[0]

        zs_sparse = is_gt * zs

        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(input)

        output = F.relu(input - taus)

        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)

        return output

In [72]:
model = Sparsemax()
output = model(gene_expressions[0])
output

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [73]:
print(torch.count_nonzero(gene_expressions[0] == 0)) 
print(torch.count_nonzero(output == 0)) 

tensor(0)
tensor(3992)


In [74]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        self.sparsemax = Sparsemax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = torch.sigmoid(self.a)  
        x = self.sparsemax(-a * x)
        return x

In [75]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.2500],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.2500],
        [0.2500],
        [0.2500]], grad_fn=<TransposeBackward0>)


In [76]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        b = torch.sigmoid(self.b)
        x = b * x
        x = self.softmax(x)
        return x


In [77]:
gene_expressions[0].shape

torch.Size([8, 500])

In [78]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output.shape)

torch.Size([8, 500])


In [79]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes


In [80]:
all_genes = pd.read_csv('./human.csv')
all_genes = all_genes['Gene'].values.tolist()

In [81]:
model = Immunogenicity(all_genes)

In [82]:
output, filtered_genes = model(current_genes)

In [83]:
len(output)

481

In [84]:

class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            z = z.unsqueeze(1)
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.sigmoid(bag_output)
            bag_outputs.append(bag_output)
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)

In [85]:
model = MIL(all_genes)

In [86]:
output = model(distances, gene_expressions, gene_names)
output

tensor([0.5668], grad_fn=<SqueezeBackward1>)

In [88]:
labels[0]


tensor(1.)

In [89]:
make_dot(output, params=dict(model.named_parameters())).render("MIL_computational_graph", format="png")


'MIL_computational_graph.png'

In [90]:
model = MIL(all_genes)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (immunogenicity): Immunogenicity()
)

In [91]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()
            output = model(distances, gene_expressions,current_genes)
            #print(output)
            #print(label)
            loss = criterion(output, label)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')


Epoch 1/5: 100%|██████████| 3858/3858 [00:10<00:00, 385.55batch/s, loss=0.568]


Epoch [1/5], Loss: 0.6848


Epoch 2/5: 100%|██████████| 3858/3858 [00:10<00:00, 379.05batch/s, loss=0.597]


Epoch [2/5], Loss: 0.6848


Epoch 3/5: 100%|██████████| 3858/3858 [00:10<00:00, 380.50batch/s, loss=0.597]


Epoch [3/5], Loss: 0.6848


Epoch 4/5: 100%|██████████| 3858/3858 [00:10<00:00, 381.59batch/s, loss=0.729]


Epoch [4/5], Loss: 0.6848


Epoch 5/5: 100%|██████████| 3858/3858 [00:10<00:00, 382.69batch/s, loss=0.838]

Epoch [5/5], Loss: 0.6847


In [97]:
def predict(model, adata, device, radius=200, max_instances=None):
    model.eval()
    
    dataset = BagsDataset(adata_radius_input=adata, immune_cell='tcell', radius=radius, max_instances=max_instances)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = np.full(len(adata.obs), np.nan)  # Initialize predictions array with NaNs

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, _, core_idx, current_genes in tepoch:
                tepoch.set_description("Predicting")
                
                # Move data to the device
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                
                output = model(distances, gene_expressions, current_genes)
                
                # Ensure we extract a single element from core_idx and output before using them
                predictions[int(core_idx.item())] = output.cpu().numpy().flatten().item()

    adata.obs['tcr_predict'] = predictions
    return adata

In [98]:
adata = predict(model, adata,radius=200, device=device)

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:00<00:00, 6272.01it/s]


Total bags created: 3858
Average instances per bag: 8


Predicting: 100%|██████████| 3858/3858 [00:04<00:00, 820.19batch/s]


In [99]:
adata.obs

,X,Y,T,cell_type,tcr_predict
AACACGTGCATCGCAC-1,7600,2200,0,0,0.531657
AACACTTGGCAAGGAA-1,4700,7100,1,1,0.567143
AACAGGATTCATAGTT-1,4900,4300,1,0,0.533965
AACAGGTTATTGCACC-1,2800,8600,0,0,0.505427
AACAGGTTCACCGAAG-1,5100,4100,1,0,0.500150
...,...,...,...,...,...
TGTTGGAACCTTCCGC-1,3500,3500,0,0,0.566660
TGTTGGAACGAGGTCA-1,2800,7200,0,1,0.551602
TGTTGGAAGCTCGGTA-1,100,9500,0,0,0.550567
TGTTGGATGGACTTCT-1,1300,5300,0,0,0.567117
